# 00 - Von OnePiece HDF zu pandas und ASE

Dieses Notebook startet direkt bei den OnePiece-HDF-Dateien. Ziel ist, die
Daten als `pandas.DataFrame` zu laden, wichtige Spalten zu verstehen und zu
zeigen, wie ASE-Objekte in solchen Tabellen typischerweise vorkommen.

**Konzept:** OnePiece ist hier die Datenbank-Schicht. Pandas ist die tabellarische
Arbeitsfläche. ASE (`ase.Atoms`) beschreibt die atomaren Strukturen, die in
DataFrame-Spalten gespeichert oder aus Dateien referenziert werden können.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

# Compatibility shim for HDF files written in a different NumPy/PyTables stack.
# Keep this cell at the top before calling pd.read_hdf.
try:
    import numpy.core as npc
    sys.modules.setdefault("numpy._core", npc)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
except Exception as exc:
    print("NumPy compatibility shim skipped:", exc)

DATA_ROOT = Path(r"/Users/dk2994/Desktop/Uni/Journal/Thesis/Notebooks/Surface Alloys")
PROJECT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI")
OUTPUT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI/notebooks/phase_diagram_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

HDF_FILES = {
    "bulk": DATA_ROOT / "CuGabulk.hdf",
    "bulk_oxide": DATA_ROOT / "CuGabulk_oxide.hdf",
    "surface_100": DATA_ROOT / "CuGasurf_100.hdf",
    "surface_110": DATA_ROOT / "CuGasurf_110.hdf",
    "surface_111": DATA_ROOT / "CuGasurf_111.hdf",
    "surface_211": DATA_ROOT / "CuGasurf_211.hdf",
}

def read_onepiece_hdf(path, key="df"):
    """Read a OnePiece-exported pandas HDF table.

    OnePiece stores simulation records in a pandas DataFrame. The HDF files in
    this tutorial are read with pd.read_hdf(filename, key="df").
    """
    path = Path(path)
    frame = pd.read_hdf(path, key=key)
    frame.attrs["source_hdf"] = str(path)
    return frame

def formula_counts(formula):
    if not isinstance(formula, str):
        return {}
    counts = {}
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts

## HDF-Dateien prüfen

In [ ]:
for label, path in HDF_FILES.items():
    print(f"{label:12s}", path.exists(), path)

## Eine OnePiece-Tabelle als DataFrame laden

In [ ]:
df_111 = read_onepiece_hdf(HDF_FILES["surface_111"])
print(df_111.shape)
df_111.head()

Ein `DataFrame` ist eine Tabelle mit Zeilen und Spalten:

- Jede Zeile ist ein Datenbankeintrag, hier z.B. eine berechnete Struktur.
- Jede Spalte ist ein Descriptor, z.B. `Name`, `Formula`, `E`,
  `form_G_per_Area`, `Ga`, `Cu` oder Koordinations-/Ladungsgrößen.
- Pandas-Befehle wie `filter`, `query`, `groupby`, `sort_values` und
  `assign` sind die wichtigsten Werkzeuge.

In [ ]:
summary = pd.DataFrame({
    "column": df_111.columns,
    "dtype": [str(df_111[c].dtype) for c in df_111.columns],
    "non_null": [int(df_111[c].notna().sum()) for c in df_111.columns],
    "example": [repr(df_111[c].dropna().iloc[0])[:90] if df_111[c].notna().any() else "" for c in df_111.columns],
})
summary

## Typische OnePiece Studio-Adapteridee

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from onepiece_studio import DataFrameSource, HDFSource, OnePieceSource

hdf_source = HDFSource(HDF_FILES["surface_111"], key="df", name="CuGa 111")
df_from_onepiece_studio = hdf_source.load()
df_from_onepiece_studio.shape

In [ ]:
class MinimalOnePieceLike:
    """A small stand-in for a OnePiece database object.

    OnePiece Studio accepts real OnePiece objects if they expose .df, .dataframe,
    or to_dataframe().
    """
    def __init__(self, df):
        self.df = df

onepiece_like = MinimalOnePieceLike(df_111)
df_from_onepiece_adapter = OnePieceSource(onepiece_like, name="minimal").load()
df_from_onepiece_adapter.head(3)

## ASE-Strukturspalten erkennen

In [ ]:
def looks_like_ase_atoms(value):
    return all(hasattr(value, attr) for attr in ["get_positions", "get_chemical_formula"])

object_columns = df_111.select_dtypes(include="object").columns.tolist()
ase_candidates = []
for column in object_columns:
    sample = df_111[column].dropna()
    if len(sample) and looks_like_ase_atoms(sample.iloc[0]):
        ase_candidates.append(column)

ase_candidates

In [ ]:
if ase_candidates:
    atoms = df_111[ase_candidates[0]].dropna().iloc[0]
    print(type(atoms))
    print(atoms.get_chemical_formula())
    print("natoms:", len(atoms))
else:
    print("No direct ase.Atoms column detected. Structures may be stored via paths or derived descriptors.")

## Erste Datenbankfragen mit pandas

In [ ]:
cols = [c for c in ["Name", "Formula", "Ga", "Cu", "Monolayer_alloy", "form_G_per_Area", "form_G_per_alloy", "E"] if c in df_111]
df_111[cols].sort_values(cols[-1]).head(10)

In [ ]:
numeric_cols = df_111.select_dtypes(include=np.number).columns
df_111[numeric_cols].describe().T.sort_values("std", ascending=False).head(15)